<a href="https://colab.research.google.com/github/geramargonzalez/AdminTriviaBuenTrato/blob/master/POC_Streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit altair scikit-learn pandas openpyxl pyngrok

In [ ]:
from pyngrok import ngrok

In [ ]:
import time, os

In [ ]:
!ngrok config add-authtoken 3Fd0aMsPtGoq8sELuEJ8Q4bHw6u_4Bg7quBYNb9H3BebDGnMi

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import hashlib
import io
from datetime import datetime
import geopandas as gpd
import geopandas as gpd
import pandas as pd

# Supponiamo di caricare i dataset di Natural Earth o fonti locali
# 1. Province dell'Argentina (Poligoni)
# URL_ADMIN1 = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_1_states_provinces.zip"
provinces = gpd.read_file("percorso_delle_province.shp")
provinces_arg = provinces[provinces["admin"] == "Argentina"].copy()

# 2. Città (Punti o Poligoni urbani) con dati sulla popolazione (es. colonna 'pop_max')
# URL_CITIES = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_populated_places.zip"
cities = gpd.read_file("percorso_delle_citta.shp")

# ---- APPROCCIO 1: Densità di popolazione all'interno della città ----
# Nota: Richiede che la geometria della città sia un poligono (Urban Areas)
# Se le città sono punti, l'area geometrica sarà 0.
if (cities.geometry.type == "Polygon").any() or (cities.geometry.type == "MultiPolygon").any():
    # Cambiamo il CRS in una proiezione metrica (es. UTMEsp32S o simile per l'Argentina, es. EPSG:5343 o 32720)
    # per calcolare accuratamente l'area in metri quadrati
    cities_metric = cities.to_crs(epsg=5343)

    # Calcolo dell'area in km² (1 km² = 1.000.000 m²)
    cities_metric["area_km2"] = cities_metric.geometry.area / 1_000_000

    # Calcolo della densità (abitanti / km²)
    cities_metric["densita_pop"] = cities_metric["pop_max"] / cities_metric["area_km2"]
    print(cities_metric[["name", "pop_max", "area_km2", "densita_pop"]].head())


# ---- APPROCCIO 2: Densità urbana aggregata per Provincia ----
# Uniamo spazialmente le città (punti) alle rispettive province
cities_with_prov = gpd.sjoin(cities, provinces_arg, how="inner", predicate="within")

# Cambiamo la proiezione della provincia in metrica per calcolarne l'area
provinces_metric = provinces_arg.to_crs(epsg=5343)
provinces_metric["area_prov_km2"] = provinces_metric.geometry.area / 1_000_000

# Aggreghiamo la popolazione urbana totale per ogni provincia
prov_pop = cities_with_prov.groupby("name_province")["pop_max"].sum().reset_index()
prov_pop.columns = ["name", "pop_urbana_totale"]

# Uniamo i dati aggregati al dataset delle province
final_provinces = provinces_metric.merge(prov_pop, on="name", how="left")

# Calcoliamo la densità di popolazione urbana per km² della provincia
final_provinces["densita_urbana_prov"] = final_provinces["pop_urbana_totale"] / final_provinces["area_prov_km2"]

# Mostra i risultati ordinati per la densità più alta
print(final_provinces[["name", "pop_urbana_totale", "area_prov_km2", "densita_urbana_prov"]].sort_values(by="densita_urbana_prov", ascending=False))







gpd = gpd.read_file("/content/provinces_features.gpkg")










Writing app.py


In [ ]:
from pyngrok import ngrok

ngrok.kill()

# Open 8501
public_url = ngrok.connect("http://localhost:8501")
print(f"Tu app está disponible en: {public_url}")

# Ejecute Streamlit
!streamlit run app.py &>/dev/null &

Tu app está disponible en: NgrokTunnel: "https://f263-35-252-149-240.ngrok-free.app" -> "http://localhost:8501"
